In [1]:
"""
GitHub Repository Data Processor and LLM Consultant

This script:
1. Fetches markdown documentation from GitHub repositories
2. Processes and chunks the content for efficient search
3. Implements text, vector, and hybrid search capabilities
4. Integrates with OpenAI API for LLM-powered responses

Usage:
    python getfilesrepo.py

Requirements:
    - Set OPENAI_API_KEY environment variable
    - Install dependencies: pip install requests python-frontmatter minsearch sentence-transformers tqdm numpy openai
"""

'\nGitHub Repository Data Processor and LLM Consultant\n\nThis script:\n1. Fetches markdown documentation from GitHub repositories\n2. Processes and chunks the content for efficient search\n3. Implements text, vector, and hybrid search capabilities\n4. Integrates with OpenAI API for LLM-powered responses\n\nUsage:\n    python getfilesrepo.py\n\nRequirements:\n    - Set OPENAI_API_KEY environment variable\n    - Install dependencies: pip install requests python-frontmatter minsearch sentence-transformers tqdm numpy openai\n'

In [15]:


import os
import io
import zipfile
from typing import List, Dict, Any
import requests
import frontmatter
from minsearch import Index, VectorSearch
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm.auto import tqdm
import openai
import pydantic_ai
import pydantic

In [9]:
# OpenAI API Configuration
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    raise ValueError("Please set the OPENAI_API_KEY environment variable")

# Model configurations
EMBEDDING_MODEL_NAME = 'multi-qa-distilbert-cos-v1'
LLM_MODEL = 'gpt-4o-mini'

# Search configurations
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 1000
NUM_SEARCH_RESULTS = 5

In [10]:
# =============================================================================
# GITHUB DATA FETCHING
# =============================================================================

def download_github_repo(repo_owner: str, repo_name: str) -> bytes:
    """
    Download a GitHub repository as a ZIP file.

    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name

    Returns:
        ZIP file content as bytes

    Raises:
        Exception: If download fails
    """
    url = f'https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main'
    response = requests.get(url)

    if response.status_code != 200:
        raise Exception(f"Failed to download repository {repo_owner}/{repo_name}: {response.status_code}")

    return response.content


def parse_markdown_files(zip_content: bytes) -> List[Dict[str, Any]]:
    """
    Parse markdown files from a ZIP archive.

    Args:
        zip_content: ZIP file content as bytes

    Returns:
        List of dictionaries containing parsed file data
    """
    repository_data = []

    with zipfile.ZipFile(io.BytesIO(zip_content)) as zf:
        for file_info in zf.infolist():
            filename = file_info.filename.lower()

            # Only process markdown files
            if not (filename.endswith('.md') or filename.endswith('.mdx')):
                continue

            try:
                with zf.open(file_info) as f:
                    content = f.read().decode('utf-8', errors='ignore')
                    # Parse frontmatter
                    post = frontmatter.loads(content)
                    data = post.to_dict()
                    data['filename'] = file_info.filename
                    repository_data.append(data)
            except Exception as e:
                print(f"Error processing {file_info.filename}: {e}")
                continue

    return repository_data


def read_repo_data(repo_owner: str, repo_name: str) -> List[Dict[str, Any]]:
    """
    Download and parse all markdown files from a GitHub repository.

    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name

    Returns:
        List of dictionaries containing file content and metadata
    """
    zip_content = download_github_repo(repo_owner, repo_name)
    return parse_markdown_files(zip_content)

def log_entry(agent, messages, source="user"):
    tools = []

    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    return {
        "agent_name": agent.name,
        "system_prompt": agent._instructions,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source
    }


In [5]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
#evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")

FAQ documents: 1285


In [11]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')

de_dtc_faq = [d for d in dtc_faq if 'data-engineering' in d['filename']]

faq_index = Index(
    text_fields=["question", "content"],
    keyword_fields=[]
)

faq_index.fit(de_dtc_faq)

In [12]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [13]:
import openai

openai_client = openai.OpenAI()

user_prompt = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
)

print(response.output_text)

It depends on the course and its enrollment policies. Many courses have specific start dates and deadlines for enrollment, while others may have rolling admissions. I suggest checking the course website or contacting the course administrator for more information on how to join now. If you have any specific details about the course, I can help you further!


In [7]:
def text_search(query):
    return faq_index.search(query, num_results=5)


# In[8]:


text_search_tool = {
    "type": "function",
    "name": "text_search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


# In[ ]:


system_prompt = """
You are a helpful assistant for a course. 
"""

question = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)

NameError: name 'openai_client' is not defined

In [17]:
import pydantic_ai
import pydantic

from typing import List, Any
#from pydantic_ai import Agent


def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the FAQ index.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the FAQ index.
    """
    return faq_index.search(query, num_results=5)


system_prompt = """
You are a helpful assistant for a  course. 

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
"""

from pydantic_ai import Agent

agent = Agent(
    name="faq_agent",
    instructions=system_prompt,
    tools=[text_search],
    model='openai:gpt-4o-mini'
)


In [10]:
question = "how do I install Kafka in Python?"
result = await agent.run(user_prompt=question)

In [18]:
import json
import secrets
from pathlib import Path
from datetime import datetime
import pydantic


LOG_DIR = Path('logs')
LOG_DIR.mkdir(exist_ok=True)


def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")


def log_interaction_to_file(agent, messages, source='user'):
    entry = log_entry(agent, messages, source)

    ts = entry['messages'][-1]['timestamp']
    ts_obj = datetime.fromisoformat(ts.replace("Z", "+00:00"))
    ts_str = ts_obj.strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath


def log_entry(agent, messages, source="user"):
    tools = []

    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    return {
        "agent_name": agent.name,
        "system_prompt": agent._instructions,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source
    }

In [19]:
question = input()
result = await agent.run(user_prompt=question)
print(result.output)
log_interaction_to_file(agent, result.new_messages())

 How do i install python


The search did not return specific instructions on how to install Python. However, I can provide general guidance on how to install Python on your system.

### Installing Python

1. **Download the Installer**:
   - Go to the official Python website: [python.org](https://www.python.org/downloads/).
   - Download the latest stable version of Python suitable for your operating system (Windows, macOS, or Linux).

2. **Run the Installer**:
   - For **Windows**:
     - Double-click the downloaded `.exe` file.
     - Make sure to check the box that says "Add Python to PATH" during installation.
     - Choose "Install Now" or customize the installation based on your preference.
   - For **macOS**:
     - Open the downloaded `.pkg` file and follow the prompts to install Python.
   - For **Linux**:
     - Open a terminal and use the package manager to install Python. For Ubuntu, for example, run:
       ```bash
       sudo apt update
       sudo apt install python3
       ```

3. **Verify Instal

NameError: name 'ModelMessagesTypeAdapter' is not defined

In [ ]:
system_prompt = """
You are a helpful assistant for a course.  

Use the search tool to find relevant information from the course materials before answering questions.  

If you can find specific information through search, use it to provide accurate answers.

Always include references by citing the filename of the source material you used.  
When citing the reference, replace "faq-main" by the full path to the GitHub repository: "https://github.com/DataTalksClub/faq/blob/main/"
Format: [LINK TITLE](FULL_GITHUB_LINK)

If the search doesn't return relevant results, let the user know and provide general guidance.  
""".strip()

# Create another version of agent, let's call it faq_agent_v2
agent = Agent(
    name="faq_agent_v2",
    instructions=system_prompt,
    tools=[text_search],
    model='gpt-4o-mini'
)


In [ ]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met. 

Checklist:

- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do  
- answer_relevant: The response directly addresses the user's question  
- answer_clear: The answer is clear and correct  
- answer_citations: The response includes proper citations or sources when required  
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: Is the search tool invoked? 

Output true/false for each check and provide a short explanation for your judgment.
""".strip()

In [ ]:
from pydantic import BaseModel

class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str

In [ ]:
eval_agent = Agent(
    name='eval_agent',
    model='gpt-5-nano',
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist
)


In [ ]:
user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

In [ ]:
def load_log_file(log_file):
    with open(log_file, 'r') as f_in:
        log_data = json.load(f_in)
        log_data['log_file'] = log_file
        return log_data

In [ ]:
log_record = load_log_file('./logs/faq_agent_v2_20250926_072928_467470.json')

instructions = log_record['system_prompt']
question = log_record['messages'][0]['parts'][0]['content']
answer = log_record['messages'][-1]['parts'][0]['content']
log = json.dumps(log_record['messages'])

user_prompt = user_prompt_format.format(
    instructions=instructions,
    question=question,
    answer=answer,
    log=log
)


In [ ]:
result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)

checklist = result.output
print(checklist.summary)

for check in checklist.checklist:
    print(check)

In [ ]:
def simplify_log_messages(messages):
    log_simplified = []

    for m in messages:
        parts = []
    
        for original_part in m['parts']:
            part = original_part.copy()
            kind = part['part_kind']
    
            if kind == 'user-prompt':
                del part['timestamp']
            if kind == 'tool-call':
                del part['tool_call_id']
            if kind == 'tool-return':
                del part['tool_call_id']
                del part['metadata']
                del part['timestamp']
                # Replace actual search results with placeholder to save tokens
                part['content'] = 'RETURN_RESULTS_REDACTED'
            if kind == 'text':
                del part['id']
    
            parts.append(part)
    
        message = {
            'kind': m['kind'],
            'parts': parts
        }
    
        log_simplified.append(message)
    return log_simplified


In [ ]:
async def evaluate_log_record(eval_agent, log_record):
    messages = log_record['messages']

    instructions = log_record['system_prompt']
    question = messages[0]['parts'][0]['content']
    answer = messages[-1]['parts'][0]['content']

    log_simplified = simplify_log_messages(messages)
    log = json.dumps(log_simplified)

    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log
    )

    result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)
    return result.output 


log_record = load_log_file('./logs/faq_agent_v2_20250926_072928_467470.json')
eval1 = await evaluate_log_record(eval_agent, log_record)


In [ ]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about a data engineering course.

Based on the provided FAQ content, generate realistic questions that students might ask.

The questions should:

- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general course questions

Generate one question for each record.
""".strip()

class QuestionsList(BaseModel):
    questions: list[str]

question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model='gpt-4o-mini',
    output_type=QuestionsList
)


In [ ]:
import random

sample = random.sample(de_dtc_faq, 10)
prompt_docs = [d['content'] for d in sample]
prompt = json.dumps(prompt_docs)

result = await question_generator.run(prompt)
questions = result.output.questions

In [ ]:
from tqdm.auto import tqdm

for q in tqdm(questions):
    print(q)

    result = await agent.run(user_prompt=q)
    print(result.output)

    log_interaction_to_file(
        agent,
        result.new_messages(),
        source='ai-generated'
    )

    print()

In [ ]:
eval_set = []

for log_file in LOG_DIR.glob('*.json'):
    if 'faq_agent_v2' not in log_file.name:
        continue

    log_record = load_log_file(log_file)
    if log_record['source'] != 'ai-generated':
        continue

    eval_set.append(log_record)

In [ ]:
eval_results = []

for log_record in tqdm(eval_set):
    eval_result = await evaluate_log_record(eval_agent, log_record)
    eval_results.append((log_record, eval_result))

In [ ]:
uv add pandas

In [ ]:
rows = []

for log_record, eval_result in eval_results:
    messages = log_record['messages']

    row = {
        'file': log_record['log_file'].name,
        'question': messages[0]['parts'][0]['content'],
        'answer': messages[-1]['parts'][0]['content'],
    }

    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)

    rows.append(row)

In [ ]:
import pandas as pd

df_evals = pd.DataFrame(rows)

In [ ]:
def evaluate_search_quality(search_function, test_queries):
    results = []
    
    for query, expected_docs in test_queries:
        search_results = search_function(query, num_results=5)
        
        # Calculate hit rate
        relevant_found = any(doc['filename'] in expected_docs for doc in search_results)
        
        # Calculate MRR
        for i, doc in enumerate(search_results):
            if doc['filename'] in expected_docs:
                mrr = 1 / (i + 1)
                break
        else:
            mrr = 0
            
        results.append({
            'query': query,
            'hit': relevant_found,
            'mrr': mrr
        })
    return results